In [ ]:
import requests

SF_USERNAME       = dbutils.secrets.get(scope="salesforce-scope", key="sf-username")
SF_PASSWORD_TOKEN = dbutils.secrets.get(scope="salesforce-scope", key="sf-password")
SF_CLIENT_ID      = "<Consumer Key from Developer Edition>"
SF_CLIENT_SECRET  = "<Consumer Secret from Developer Edition>"

response = requests.post(
    "https://login.salesforce.com/services/oauth2/token",
    data={
        "grant_type"   : "password",
        "client_id"    : SF_CLIENT_ID,
        "client_secret": SF_CLIENT_SECRET,
        "username"     : SF_USERNAME,
        "password"     : SF_PASSWORD_TOKEN  # password+security token combined
    }
)

data = response.json()
print("Refresh Token:", data.get("refresh_token"))
print("Instance URL :", data.get("instance_url"))

if "error" in data:
    print(" Error:", data)

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import ConnectionType

w     = WorkspaceClient()
SCOPE = "salesforce-scope"

connection = w.connections.create(
    name="salesforce_azure_conn",
    connection_type=ConnectionType.SALESFORCE,
    options={
        "client_id"    : dbutils.secrets.get(scope=SCOPE, key="sf-client-id"),
        "client_secret": dbutils.secrets.get(scope=SCOPE, key="sf-client-secret"),
        "refresh_token": dbutils.secrets.get(scope=SCOPE, key="sf-refresh-token"),
        "is_sandbox"   : "false"
    }
)
print(f"Connection created: {connection.name}")

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import ConnectionType

w     = WorkspaceClient()
SCOPE = "salesforce-scope"

connection = w.connections.create(
    name="salesforce_azure_conn",
    connection_type=ConnectionType.SALESFORCE,
    options={
        "client_id"    : dbutils.secrets.get(scope=SCOPE, key="sf-client-id"),
        "client_secret": dbutils.secrets.get(scope=SCOPE, key="sf-client-secret"),
        "refresh_token": dbutils.secrets.get(scope=SCOPE, key="sf-refresh-token"),
        "is_sandbox"   : "false"
    }
)
print(f" Connection created: {connection.name}")

In [ ]:
data_engineering_workshop.sf_data

In [ ]:
from databricks.sdk.service.pipelines import (
    IngestionPipelineDefinition,
    IngestionConfig,
    TableSpec,
)

# Create pipeline
pipeline = w.pipelines.create(
    name="sfdc_account_azure",
    catalog="data_engineering_workshop",
    target="sf_data",
    ingestion_definition=IngestionPipelineDefinition(
        connection_name="salesforce_azure_conn",
        objects=[
            IngestionConfig(
                table=TableSpec(
                    source_schema="objects",
                    source_table="Account",
                    destination_catalog="data_engineering_workshop",
                    destination_schema="sf_data",
                )
            )
        ]
    ),
    continuous=False,
)

pipeline_id = pipeline.pipeline_id
print(f"Pipeline created! ID: {pipeline_id}")

In [ ]:
import time

run       = w.pipelines.start_update(pipeline_id=pipeline_id, full_refresh=False)
update_id = run.update_id
print(f"Run triggered! Update ID: {update_id}")

while True:
    status = w.pipelines.get_update(pipeline_id=pipeline_id, update_id=update_id)
    state  = status.update.state.value
    print(f"   State: {state}")
    if state in ("COMPLETED", "FAILED", "CANCELED"):
        break
    time.sleep(15)

print(f"{' Done!' if state == 'COMPLETED' else ' Failed: ' + state}")

In [ ]:
df = spark.table("data_engineering_workshop.sf_data.Account")
print(f"Total Account records: {df.count()}")
#display(df.limit(10))